In [1]:
import ROOT
ROOT.EnableImplicitMT(16)
import pandas as pd
import libPy
weights = [
    "hw_nominal",     # nominal MC weight.                  scalar
    "hw_alphaS_up",   # up alpha_s variaton for PHD4LHC;    scalar
    "hw_alphaS_dn",   # down alpha_s variaton for PHD4LHC;  scalar
    "hw_pdf4lhc_unc", # 30 Eigen variation for PHD4LHC      vector
    "hw_qcd",         # muR/muF variation for the given MC; vector
]

In [2]:
dfs = {}
dfs['ALL']      = ROOT.RDataFrame("tree", "ntuples/mc20_vbf_hyy_stxs.root")
dfs['UNKNOWN']  = dfs['ALL'].Filter("HTXS_Stage1_2_Fine_Category_pTjet30 == 0")

for val, category in libPy.stage_1_2_fine['vbf'].items():
    dfs[category] = dfs['ALL'].Filter(f"HTXS_Stage1_2_Fine_Category_pTjet30 == {val}")

# # determine the length of the vector weights, which should be the same for all events
# len_hw_pdf4lhc_unc = set(dfs['ALL'].Range(10).Define('len_hw_pdf4lhc_unc', 'hw_pdf4lhc_unc.size()').AsNumpy(['len_hw_pdf4lhc_unc'])['len_hw_pdf4lhc_unc'])
# len_hw_qcd         = set(dfs['ALL'].Range(10).Define('len_hw_qcd', 'hw_qcd.size()').AsNumpy(['len_hw_qcd'])['len_hw_qcd'])
# assert (len(len_hw_pdf4lhc_unc) == 1 and len(len_hw_qcd) == 1)
# len_hw_pdf4lhc_unc = list(len_hw_pdf4lhc_unc)[0]
# len_hw_qcd         = list(len_hw_qcd)[0]
len_hw_pdf4lhc_unc, len_hw_qcd = 30, 8

In [3]:
weight_dict = {}
futures = []
for slice, df in dfs.items():
    weight_dict[slice] = {}
    for weight in weights:
        if weight == "hw_pdf4lhc_unc":
            for i in range(len_hw_pdf4lhc_unc):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        elif weight == "hw_qcd":
            for i in range(len_hw_qcd):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        else:
            weight_dict[slice][weight] = df.Filter(f"{weight} == {weight}").Sum(weight)
            futures.append(weight_dict[slice][weight])
ROOT.RDF.RunGraphs(futures)

1

In [4]:
for slice, weight_sum_dict in weight_dict.items():
    for weight_name, weight_sum in weight_sum_dict.items():
        weight_dict[slice][weight_name] = weight_sum.GetValue()

In [5]:
pdf = pd.DataFrame(weight_dict)
pdf = pdf.apply(lambda row : (row / row['ALL']), axis=1)
ratio_pdf = pdf.apply(lambda row : row / pdf.iloc[0], axis=1)
# pdf.columns = [col + '_acc' for col in pdf.columns]
# pdf[[col.split('_')[0] + '_xs' for col in pdf.columns]] = pdf.apply(lambda row : row * xs, axis=1)
# pdf = pd.concat([pdf, ratio_pdf.add_suffix('_ratio')], axis=1)
pdf.to_csv("res_stxs/run2_vbf_stxs.csv", index=True)

In [9]:
import pandas as pd
pdf = pd.read_csv("res_stxs/run2_vbf_stxs.csv", index_col=0)

official = {key.upper() : val for key, val in libPy.official_1_2_fine_run2['vbf'].items()}
official['UNKNOWN'] = 100 - sum(official.values())
mine = pdf.iloc[0] * 100
mine.index = [index.upper() for index in mine.index]
comp_df = pd.DataFrame.from_dict(official, orient='index')
comp_df.columns = ['official']
comp_df['mine'] = mine
comp_df['diff'] = comp_df['mine'] - comp_df['official']
comp_df['diff_pct'] = abs(comp_df['diff'] / comp_df['official'] * 100)
comp_df

,official,mine,diff,diff_pct
QQ2HQQ_FWDH,7.035200,7.014432,-0.020768,0.295207
QQ2HQQ_0J,7.392270,7.377265,-0.015005,0.202982
QQ2HQQ_1J,34.197800,34.238375,0.040575,0.118649
QQ2HQQ_GE2J_MJJ_0_60_PTHJJ_0_25,0.505031,0.500756,-0.004275,0.846440
QQ2HQQ_GE2J_MJJ_60_120_PTHJJ_0_25,0.997949,1.011414,0.013465,1.349258
QQ2HQQ_GE2J_MJJ_120_350_PTHJJ_0_25,7.549220,7.521600,-0.027620,0.365870
QQ2HQQ_GE2J_MJJ_0_60_PTHJJ_GT25,0.808783,0.810754,0.001971,0.243668
QQ2HQQ_GE2J_MJJ_60_120_PTHJJ_GT25,1.285060,1.284207,-0.000853,0.066395
QQ2HQQ_GE2J_MJJ_120_350_PTHJJ_GT25,3.642060,3.654628,0.012568,0.345077
QQ2HQQ_GE2J_MJJ_350_700_PTH_0_200_PTHJJ_0_25,10.314500,10.299265,-0.015235,0.147703
